# Methodology 2 (Diverse Ensemble) — Mistral-7B on Colab

Runs the 3-strategy ensemble (Chain-of-Thought + Formula-First + Backward -> majority vote) on **Mistral-7B-Instruct-v0.3**, SVAMP **N=300**. Standalone, **resumable from Google Drive**.

**Runtime -> Change runtime type -> T4 GPU.** Accept the Mistral license first: https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.3

## 1. Install dependencies

In [ ]:
!nvidia-smi
!pip -q install -U bitsandbytes transformers accelerate datasets sentencepiece huggingface_hub
import torch; print('cuda:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'No GPU! Runtime -> Change runtime type -> T4 GPU'

## 2. Mount Drive (results persist here -> resume-safe)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
OUT_ROOT='/content/drive/MyDrive/SLM_A1/outputs_ensemble_n300'
SLUG='mistral-7b'
OUT_DIR=f'{OUT_ROOT}/{SLUG}'; os.makedirs(OUT_DIR, exist_ok=True)
print('writing to', OUT_DIR)

## 3. Hugging Face login (Mistral is gated)

In [ ]:
from huggingface_hub import login
login()  # paste your HF read token

## 4. Run the ensemble (Mistral, N=300). Resumes from Drive.

In [ ]:
import json, re, time, os
from collections import Counter
import logging
logging.getLogger("bitsandbytes.autograd._functions").setLevel(logging.ERROR)
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from datasets import load_dataset

MODEL_ID="mistralai/Mistral-7B-Instruct-v0.3"
N=300; MAX_NEW=512
STRATEGIES=["chain_of_thought","formula_first","backward"]

# ---- data (SVAMP, seed-42 shuffle: first 100 of 300 == published subset) ----
def load_svamp(n, seed=42):
    ds=load_dataset("ChilleD/SVAMP", split="train").shuffle(seed=seed)
    rows=[]
    for it in ds.select(range(min(n,len(ds)))):
        body=str(it.get("Body","")).strip(); q=str(it.get("Question","")).strip()
        rows.append({"question":f"{body} {q}".strip(),"answer":str(it["Answer"]).strip()})
    return rows

_TAIL=("You MUST end your response with exactly this format: FINAL_ANSWER: [your numerical answer]\n"
       "Do not write placeholders. Write the actual number or expression as your answer.\n"
       "Do not ask follow-up questions. Do not invite further input. Produce a complete solution now.\n\n")
PROMPTS={
 "chain_of_thought": lambda q: ("Solve the following problem by thinking step by step.\n"
    "Show each reasoning step clearly and in order.\n"+_TAIL+f"Problem: {q}\n\nStep-by-step solution:"),
 "formula_first": lambda q: ("Solve the following problem using a formula-first approach.\n"
    "First, write out every equation or formula you will need.\n"
    "Then substitute values and compute the result.\n"+_TAIL+f"Problem: {q}\n\nEquations and solution:"),
 "backward": lambda q: ("Solve the following problem using backward reasoning.\n"
    "Start by clearly stating what quantity the question asks for.\n"
    "Work backwards: what do you need to compute that? What do you need before that?\n"+_TAIL+f"Problem: {q}\n\nBackward reasoning:"),
}

_FA=re.compile(r"FINAL_ANSWER\s*:\s*(.+)",re.I); _PH=re.compile(r"^(<[^>]*>|\[[^\]]*\])$")
_NUM=re.compile(r"[-+]?\d[\d,.]*"); _NUMF=re.compile(r"[-+]?\d+(?:\.\d+)?"); _FRAC=re.compile(r"\\frac\{([^{}]+)\}\{([^{}]+)\}")
def extract_final_answer(text):
    m=_FA.search(text)
    if m:
        c=m.group(1).strip()
        if not _PH.match(c): return c
    nums=_NUM.findall(text)
    if nums: return nums[-1].replace(",","")
    lines=[l.strip() for l in text.splitlines() if l.strip()]
    return lines[-1] if lines else text.strip()
def _norm(t):
    t=t.strip(); b=re.search(r"\\boxed\{(.+)\}",t,re.DOTALL)
    if b: t=b.group(1)
    t=t.strip().lower(); t=re.sub(r"[$,%]","",t); t=re.sub(r"\s+"," ",t); return t.rstrip(".").strip()
def _f(t):
    try: return float(t)
    except ValueError: return None
def _fr(t):
    m=_FRAC.match(t.strip())
    if m:
        try: return float(m.group(1))/float(m.group(2))
        except (ValueError,ZeroDivisionError): return None
    return None
def is_correct(p,g):
    a,b=_norm(p),_norm(g)
    if a==b: return True
    af,bf=_f(a),_f(b)
    if af is not None and bf is not None: return abs(af-bf)<1e-6
    ar,br=_fr(a),_fr(b)
    if ar is not None and br is not None: return abs(ar-br)<1e-6
    if ar is not None and bf is not None: return abs(ar-bf)<1e-6
    if br is not None and af is not None: return abs(af-br)<1e-6
    nums=_NUMF.findall(a)
    if nums:
        lf=_f(nums[-1])
        if lf is not None and bf is not None: return abs(lf-bf)<1e-6
    return False
def summarize(rows):
    total=len(rows); correct=sum(r["correct"] for r in rows)
    tok=sum(int(r["token_cost"]) for r in rows); avg=tok/total if total else 0.0; acc=correct/total if total else 0.0
    s={"num_examples":total,"num_correct":correct,"accuracy":round(acc,4),"total_token_cost":tok,
       "avg_token_cost":round(avg,2),"nate":round((acc/avg)*1000,6) if avg else 0.0}
    if rows and rows[0].get("strategy_predictions"):
        strat=list(rows[0]["strategy_predictions"].keys())
        for st in strat:
            sc=sum(is_correct(r["strategy_predictions"].get(st,""),r.get("gold_answer","")) for r in rows)
            s[f"accuracy_{st}"]=round(sc/total,4)
        if len(strat)>1:
            s["disagreement_rate"]=round(sum(1 for r in rows if len(set(r["strategy_predictions"].values()))>1)/total,4)
            s["unique_correct_rate"]=round(sum(1 for r in rows if sum(is_correct(r["strategy_predictions"].get(st,""),r.get("gold_answer","")) for st in strat)==1)/total,4)
    return s

# ---- load model ----
print("Loading", MODEL_ID, "...")
tok=AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
if tok.pad_token is None: tok.pad_token=tok.eos_token
bnb=BitsAndBytesConfig(load_in_8bit=True)
model=AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb, device_map="auto", torch_dtype=torch.float16, trust_remote_code=True)
def generate(prompt):
    maxlen=getattr(model.config,"max_position_embeddings",2048)
    enc=tok(prompt,return_tensors="pt",truncation=True,max_length=maxlen)
    enc={k:v.to("cuda") for k,v in enc.items()}
    with torch.inference_mode():
        out=model.generate(**enc,max_new_tokens=MAX_NEW,do_sample=False,pad_token_id=tok.pad_token_id,eos_token_id=tok.eos_token_id)
    ilen=enc["input_ids"].shape[1]; comp=out[0][ilen:]
    return {"text":tok.decode(comp,skip_special_tokens=True).strip(),"prompt_tokens":int(ilen),"completion_tokens":int(comp.shape[0])}

data=load_svamp(N); print("Loaded", len(data), "examples")
pred_path,summ_path=f"{OUT_DIR}/predictions.json",f"{OUT_DIR}/summary.json"
rows,done=[],set()
if os.path.exists(pred_path):
    try: rows=json.load(open(pred_path)); done={r["index"] for r in rows}; print("resume", len(done))
    except Exception: rows,done=[],set()
t0=time.time()
for idx,ex in enumerate(data):
    if idx in done: continue
    texts,preds,toks={}, {}, {}
    try:
        for st in STRATEGIES:
            o=generate(PROMPTS[st](ex["question"]))
            texts[st]=o["text"]; preds[st]=extract_final_answer(o["text"]); toks[st]=o["prompt_tokens"]+o["completion_tokens"]
        final=Counter(preds.values()).most_common(1)[0][0]
        result={"strategy_texts":texts,"strategy_predictions":preds,"prediction":final,
                "correct":is_correct(final,ex["answer"]),"token_cost":sum(toks.values()),
                "per_strategy_tokens":toks,"aggregation":"majority_vote"}
    except Exception as e:
        print(idx+1,"ERROR",e)
        result={"strategy_texts":{},"strategy_predictions":{},"prediction":"","correct":False,
                "token_cost":0,"per_strategy_tokens":{},"aggregation":"error","error":str(e)}
    rows.append({"index":idx,"question":ex["question"],"gold_answer":ex["answer"],**result})
    if (idx+1)%10==0 or (idx+1)==len(data):
        rows.sort(key=lambda r:r["index"]); json.dump(rows,open(pred_path,"w"),indent=2,ensure_ascii=False)
        acc=sum(r["correct"] for r in rows)/len(rows)
        print(f"[mistral-7b] {idx+1}/{len(data)} acc={acc:.3f} {(idx+1-len(done))/max(time.time()-t0,1e-6):.3f}it/s")
rows.sort(key=lambda r:r["index"]); json.dump(rows,open(pred_path,"w"),indent=2,ensure_ascii=False)
s=summarize(rows); json.dump(s,open(summ_path,"w"),indent=2,ensure_ascii=False)
print("DONE:", s)

## 5. (Optional) download a zip; results already saved to Drive

In [ ]:
import shutil
z=shutil.make_archive('/content/mistral-7b_ensemble_n300','zip',OUT_ROOT)
from google.colab import files; files.download(z)